# Lab 14: Segmentation

**Module 14** | Read `notes/14-segmentation-unet-maskrcnn.pdf` before running this notebook.

Run a pre-trained DeepLabV3 on a sample image and visualize semantic segmentation masks.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Semantic segmentation with DeepLabV3

**Semantic segmentation** assigns a class label to *every pixel* in an image. Detection draws boxes around objects; segmentation paints each pixel as "person," "sky," "car," and so on.

**DeepLabV3** uses atrous (dilated) convolutions to capture context at multiple scales, then outputs a dense label map. The ResNet-50 variant pre-trained on COCO produces 21 Pascal-VOC-style classes (background plus 20 objects).

This lab runs inference, prints a per-class pixel histogram, identifies dominant classes, colorizes the mask, and shows input versus segmentation side by side.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **Semantic segmentation** | Label every pixel with a class (road, person, sky) rather than drawing bounding boxes around objects. |
| **Dense prediction** | The network outputs a full H-by-W label map, not just a single label per image. |
| **Argmax** | Pick the class with the highest score at each pixel. Converts per-class logits to one label per pixel. |
| **VOC classes** | PASCAL VOC object categories (20 objects plus background) used by this pre-trained head. |
| **Bounding box** | Not used in segmentation output, but detection labs (12 and 13) use boxes instead of per-pixel labels. |

Refer back to this table whenever a term appears in code or output.


### Step 1: Load model and sample image

**What this section does:** Loads DeepLabV3 with ResNet-50 weights, finds sample JPGs, and prints all 21 VOC class names.

**Why we do it:** The weight metadata includes the exact preprocessing transforms and label names. Printing class indices now makes the histogram in the next step easier to read.


**What to look for in the output**

- Model name and class count (21).
- Input image filename and dimensions.
- VOC class list from index 0 (background) through 20.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torchvision.models.segmentation import (
    DeepLabV3_ResNet50_Weights,
    deeplabv3_resnet50,
)
from runtime_env import ensure_sample_images

ROOT = Path("..").resolve()
IMAGE_DIR = ensure_sample_images()

weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = deeplabv3_resnet50(weights=weights)
model.eval().to(device)
preprocess = weights.transforms()
class_names = weights.meta["categories"]

image_paths = sorted(IMAGE_DIR.glob("*.jpg"))
if not image_paths:
    raise FileNotFoundError(
        f"No JPG in {IMAGE_DIR}. Check your network connection or run: python download_datasets.py"
    )
img_path = image_paths[0]
original = Image.open(img_path).convert("RGB")

print(f"Model: DeepLabV3 ResNet-50 (VOC classes)")
print(f"Number of classes: {len(class_names)}")
print(f"Input image: {img_path.name} ({original.size[0]}x{original.size[1]})")
print()
print("VOC class labels:")
for idx, name in enumerate(class_names):
    print(f"  {idx:2d}: {name}")


### Step 2: Run inference and pixel histogram

**What this section does:** Forward-passes the preprocessed image, takes `argmax` over class logits to get a per-pixel label map, and counts how many pixels belong to each class.

**Why we do it:** The histogram tells you scene composition (often lots of background or sky). It is a quick sanity check before viewing the colored mask.


**What to look for in the output**

- Segmentation map shape matches the network output resolution (may differ from original image size).
- Unique class ids listed (subset of 0-20).
- Histogram rows only for classes with at least one pixel.
- TOTAL row sums to 100% of pixels.


In [ ]:
batch = preprocess(original).unsqueeze(0).to(device)
with torch.no_grad():
    out = model(batch)["out"]

# argmax picks the winning class at each spatial location.
mask = out.argmax(dim=1).squeeze(0).cpu().numpy()
unique_ids = np.unique(mask)

print(f"Segmentation map shape: {mask.shape}")
print(f"Unique class ids in scene: {unique_ids.tolist()}")
print()

counts = np.bincount(mask.flatten(), minlength=len(class_names))
total_pixels = mask.size

print("Class histogram (pixel counts):")
print("=" * 55)
print(f"{'ID':>3}  {'Class':<18}  {'Pixels':>10}  {'Percent':>8}")
print("-" * 55)
for cls_id in range(len(class_names)):
    if counts[cls_id] > 0:
        pct = 100.0 * counts[cls_id] / total_pixels
        print(f"{cls_id:3d}  {class_names[cls_id]:<18}  {counts[cls_id]:>10,}  {pct:7.2f}%")
print("-" * 55)
print(f"{'':3}  {'TOTAL':<18}  {total_pixels:>10,}  {'100.00%':>8}")


### Step 3: Top classes and detected VOC labels

**What this section does:** Ranks classes by pixel count and lists which VOC object classes (excluding background) appear in the scene.

**Why we do it:** Large uniform regions (background, sky) often dominate pixel counts. The detected-object list highlights actual things in the scene.


**What to look for in the output**

- Top 8 classes printed with pixel counts and percentages.
- VOC object classes detected count (may be 0 on plain scenes).
- Comma-separated list of object names if any are present.


In [ ]:
nonzero = [(i, counts[i]) for i in range(len(class_names)) if counts[i] > 0]
ranked = sorted(nonzero, key=lambda x: x[1], reverse=True)

print("Top classes in scene (by pixel count):")
for rank, (cls_id, count) in enumerate(ranked[:8], start=1):
    pct = 100.0 * count / total_pixels
    print(f"  {rank}. {class_names[cls_id]:<18} {count:>8,} px ({pct:.2f}%)")

detected_objects = [
    class_names[i] for i in unique_ids if i != 0 and class_names[i] != "__background__"
]
print()
print(f"VOC object classes detected (excluding background): {len(detected_objects)}")
if detected_objects:
    print("  " + ", ".join(detected_objects))
else:
    print("  (none, scene may be mostly background or unlabeled regions)")


### Step 4: Colorize and compare

**What this section does:** Maps each class id to a random RGB color (fixed seed for reproducibility) and shows the original image next to the colored mask.

**Why we do it:** Human eyes parse color maps faster than integer label grids. Background is forced to black so foreground regions stand out.


**What to look for in the output**

- Side-by-side figure with input on the left and colored segmentation on the right.
- Colored regions should roughly align with objects or surfaces in the photo.
- Background areas appear black in the mask panel.


In [ ]:
rng = np.random.default_rng(42)
num_classes = out.shape[1]
palette = rng.integers(0, 255, size=(num_classes, 3), dtype=np.uint8)
palette[0] = [0, 0, 0]  # Background stays black.

colored = palette[mask]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original)
axes[0].set_title(f"Input: {img_path.name}")
axes[0].axis("off")

axes[1].imshow(colored)
axes[1].set_title("DeepLabV3 segmentation (colored classes)")
axes[1].axis("off")

plt.tight_layout()
plt.show()


### Step 5: Evaluation summary

**What this section does:** Prints image name, map resolution, class counts, and the dominant class.

**Why we do it:** Segmentation errors often appear at object boundaries or on small instances that occupy few pixels. Compare the dominant class to what you see in the side-by-side plot.


**What to look for in the output**

- Map resolution matches Step 2.
- Classes with pixels count matches histogram.
- Dominant class name and percentage look reasonable for the image content.


In [ ]:
print("Segmentation evaluation:")
print(f"  Image: {img_path.name}")
print(f"  Map resolution: {mask.shape[0]}x{mask.shape[1]}")
print(f"  Classes with pixels: {len(nonzero)}")
print(f"  VOC objects detected: {detected_objects}")
print(f"  Dominant class: {class_names[ranked[0][0]]} ({100.0 * ranked[0][1] / total_pixels:.1f}% of pixels)")
